<h1>Importing</h1>

In [44]:
import pandas as pd
import numpy as np
import os
import chromadb 
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from sentence_transformers import SentenceTransformer
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

<h1>setting up </h1>

In [6]:
MODEL_NAME = "all-MiniLM-L6-v2"

<h1>Dataset & Embedding functions</h1>

In [3]:
descrs = pd.read_parquet('./dataset/dataset.parquet').description
prices = pd.read_parquet('./dataset/dataset.parquet').Price
links = pd.read_excel('./dataset/dataset.xlsx').url
meta = []
for prc, link, id in zip(prices, links, prices.index):
  meta.append({'Price':prc, 'Url':link, 'Id':id})

In [4]:
chroma_client = chromadb.PersistentClient(path="./chroma_db2")



collection = chroma_client.get_or_create_collection(
    name="descriptions",
    embedding_function=SentenceTransformerEmbeddingFunction(model_name=MODEL_NAME),
    metadata={"hnsw:space": "cosine"}
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6640.23it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
HTTP Error 504 thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json
Retrying in 1s [Retry 1/5].
HTTP Error 503 thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json
Retrying in 2s [Retry 2/5].
HTTP Error 503 thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json
Retrying in 4s [Retry 3/5].
HTTP Error 503 thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniL

<h1>Loading the descriptions (w/o batches)</h1>

In [5]:
collection.add(
  documents=descrs.values.tolist(),
  metadatas=meta,
  ids=np.vectorize(str)(descrs.index).tolist(),
)

In [7]:
print(f'Total Descriptions loaded: {collection.count()}')

Total Descriptions loaded: 2111


In [8]:
sample = collection.get(ids='100', include=["documents", "metadatas"])
sample

{'ids': ['100'],
 'embeddings': None,
 'documents': ["Experience retro flair and modern luxury at The Arroyo House, a freshly renovated 1969 atomic ranch-style home. Nestled in exclusive South Pasadena, this single-story abode serves as the perfect launching pad for your next Los Angeles adventure. Walk, bike, or drive to historic downtown South Pasadena featuring cafes, museums, weekly farmer’s market, specialty stores, L.A. Metro Rail station, and more. The Rose Bowl is accessible by an adjacent hiking trail or a five-minute drive from the home.The spaceEmbracing its unique past as a music school, The Arroyo House is a haven for music enthusiasts. The home is adorned with musical memorabilia and instruments, including a Lyon & Healy baby grand piano, acoustic guitars, a ukulele, Toca congas, and other instruments. Immerse yourself in the rhythm with a retro turntable and a rotating curated collection of vinyl records from the owners' personal stash. Indulge in the breathtaking views 

<h1>Embedding searching</h1>

In [9]:
def search(query: str, conditions: dict = None) -> list[str]:
  results = collection.query(
    query_texts=[query],
    n_results = 100,
    where=conditions,
    include=["documents", "metadatas", "distances"]
  )
  return results
  
QUERY = 'A lodge or a remote house to get away from the hustle and bustle of daily life for a few days with fine scenery to behold'
results = search(QUERY)


In [11]:
print(f'The DESCR of the FIRST search result: {results['documents'][0][0]}')
print(f'The 5-DAY PRICE of the FIRST search result: {results['metadatas'][0][0]['Price']}')
print(f'The URL of the FIRST search result: {results['metadatas'][0][0]['Url']}')
print(f'The DATASET ID of the FIRST search result: {results['metadatas'][0][0]['Id']}')

The DESCR of the FIRST search result: **Welcome to Your Ideal Family Getaway!Discover the perfect retreat in our charming one-story home, designed with families and groups in mind. Located in a quiet, family-friendly neighborhood, this spacious residence features four large bedrooms, ensuring everyone has their own comfortable space. Ideal for relaxation after a day of exploration, this home combines modern amenities with a cozy atmosphere, making it an inviting sanctuary for you. W/ modern & inviting open concept,3 dining areas.The spaceAs you enter, you’ll be greeted by a warm and welcoming ambiance. The open-concept living area is thoughtfully furnished with your comfort in mind, featuring plush seating and a cozy fireplace—perfect for gathering around during chilly evenings. Enjoy movie nights with loved ones on one of the five TVs available throughout the house, providing entertainment options for everyone.The fully equipped kitchen invites you to whip up delicious meals, with amp

<h1>Searching + NN prompting</h1>

In [78]:
TEMPLATE = ChatPromptTemplate.from_messages([
  ('system', """You are an analysis assistant. 
   You will be given information regarding AirBNB rental house listings.
   For each separate listing, you must decide whether the Price is fair according to its own description and the prices and descriptions of other listings.
   Your end goal is to select the best deals among the provided listings.
   Your analysis must be based only on the provided context. Be concise."""),
  ('human', """
   A single listing takes on the following text format:
   ID: [listing id] | Price: [listing price] | Description : [description]
   
   Here is the list of listings to analyze:
   {listings_list}
   
   Please, in your answer, provide the following bullet points:
   - Suggest 6 best deals among the provided listings. Provide their ID.
   - Select listings that are, objectively, overpriced (if any, otherwise, clearly state that such listings do not exist in the given list)
   - For each of the 6 listings that you've defined as "best deals", provide the reasoning behind your choice.
   """)])

<h2>Сначала скачать Ollama -> ollama pull mistral:instruct -> ollama run mistral:instruct</h2>

In [79]:

def NN_search(semantic_search_query: str, result_count: int = 20, semantic_search_conditions: dict = None) -> str:
  lst = []
  results = collection.query(
    query_texts=[semantic_search_query],
    n_results = result_count,
    where=semantic_search_conditions,
    include=["documents", "metadatas"]
  )
  for ind, descr, price in zip(results['ids'][0], results['documents'][0], results['metadatas'][0]):
    lst.append(f'ID: {ind} | Price: {price['Price']} | Description: {descr}')

  llm = ChatOllama(
    model = 'mistral:instruct',
    temperature=0.7
  )

  
  text = '\n'.join(lst)
  prompt = TEMPLATE.format_messages(listings_list = text)
  
  response = llm.invoke(prompt)
  
  return response.content
  


<h3>Никак не смог заставить Мистраль не вставлять валюту</h3>

In [82]:
answer = NN_search(QUERY).replace('$', '')

In [85]:
print('\n'.join(answer.split('\n')))

 - Best Deals:
  1. ID: 70 - The listing offers a beautiful and renovated craftsman home with stunning views, high-speed internet, and multiple outdoor patios for relaxation. The price is reasonable considering its location and amenities. (Price: 132520)
  2. ID: 884 - This vacation home provides breathtaking water and mountain views, a peaceful location, and a calming environment. The price seems fair given the tranquility and picturesque setting it offers. (Price: 141330)
  3. ID: 857 - A cozy and affordable studio apartment located in the heart of the city with high-speed internet, a comfortable bed, and a well-equipped kitchenette. Ideal for budget travelers or those looking for a short stay in the city. (Price: 64020)
  4. ID: 135 - A spacious one-bedroom apartment offering panoramic views of the city skyline, modern amenities, and a comfortable living space. The price is competitive compared to similar listings in the area. (Price: 89670)
  5. ID: 452 - This listing offers a char